# Joins

Notebook version of `joins.py` so each step can be rerun independently while working through the lesson.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Joins Notebook")
    .config("spark.master", "local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/17 22:49:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
guitars_df = spark.read.json("src/main/resources/data/guitars.json")
guitarists_df = spark.read.json("src/main/resources/data/guitarPlayers.json")
bands_df = spark.read.json("src/main/resources/data/bands.json")

guitars_df.show(truncate=False)
guitarists_df.show(truncate=False)
bands_df.show(truncate=False)

+----------------------+---+------+------------+
|guitarType            |id |make  |model       |
+----------------------+---+------+------------+
|Electric double-necked|0  |Gibson|EDS-1275    |
|Electric              |5  |Fender|Stratocaster|
|Electric              |1  |Gibson|SG          |
|Acoustic              |2  |Taylor|914         |
|Electric              |3  |ESP   |M-II        |
+----------------------+---+------+------------+

+----+-------+---+------------+
|band|guitars|id |name        |
+----+-------+---+------------+
|0   |[0]    |0  |Jimmy Page  |
|1   |[1]    |1  |Angus Young |
|2   |[1, 5] |2  |Eric Clapton|
|3   |[3]    |3  |Kirk Hammett|
+----+-------+---+------------+

+-----------+---+------------+----+
|hometown   |id |name        |year|
+-----------+---+------------+----+
|Sydney     |1  |AC/DC       |1973|
|London     |0  |Led Zeppelin|1968|
|Los Angeles|3  |Metallica   |1981|
|Liverpool  |4  |The Beatles |1960|
+-----------+---+------------+----+



In [3]:
# Inner join: only matching rows from both sides are kept.
join_condition = guitarists_df["band"] == bands_df["id"]
guitarists_bands_df = guitarists_df.join(bands_df, join_condition, "inner")

guitarists_bands_df.show()

+----+-------+---+------------+-----------+---+------------+----+
|band|guitars| id|        name|   hometown| id|        name|year|
+----+-------+---+------------+-----------+---+------------+----+
|   1|    [1]|  1| Angus Young|     Sydney|  1|       AC/DC|1973|
|   0|    [0]|  0|  Jimmy Page|     London|  0|Led Zeppelin|1968|
|   3|    [3]|  3|Kirk Hammett|Los Angeles|  3|   Metallica|1981|
+----+-------+---+------------+-----------+---+------------+----+



In [ ]:
# Left outer join: inner join and LEFT table
# # keep everything from the guitarists side, fill missing band data with nulls.
# Includes eric clapton when he has no band
guitarists_df.join(bands_df, join_condition, "left_outer").show()

+----+-------+---+------------+-----------+----+------------+----+
|band|guitars| id|        name|   hometown|  id|        name|year|
+----+-------+---+------------+-----------+----+------------+----+
|   0|    [0]|  0|  Jimmy Page|     London|   0|Led Zeppelin|1968|
|   1|    [1]|  1| Angus Young|     Sydney|   1|       AC/DC|1973|
|   2| [1, 5]|  2|Eric Clapton|       NULL|NULL|        NULL|NULL|
|   3|    [3]|  3|Kirk Hammett|Los Angeles|   3|   Metallica|1981|
+----+-------+---+------------+-----------+----+------------+----+



In [5]:
# right outer join - everything in inner join + everything in RIGHT table
# should include beatles that has no guitarist
guitarists_df.join(bands_df, join_condition, "right_outer").show()

+----+-------+----+------------+-----------+---+------------+----+
|band|guitars|  id|        name|   hometown| id|        name|year|
+----+-------+----+------------+-----------+---+------------+----+
|   1|    [1]|   1| Angus Young|     Sydney|  1|       AC/DC|1973|
|   0|    [0]|   0|  Jimmy Page|     London|  0|Led Zeppelin|1968|
|   3|    [3]|   3|Kirk Hammett|Los Angeles|  3|   Metallica|1981|
|NULL|   NULL|NULL|        NULL|  Liverpool|  4| The Beatles|1960|
+----+-------+----+------------+-----------+---+------------+----+



In [ ]:
# full outer join = everything in inner and BOTH tables
# will have eric clapton and the beatles
guitarists_df.join(bands_df, join_condition, "outer").show()

+----+-------+----+------------+-----------+----+------------+----+
|band|guitars|  id|        name|   hometown|  id|        name|year|
+----+-------+----+------------+-----------+----+------------+----+
|   0|    [0]|   0|  Jimmy Page|     London|   0|Led Zeppelin|1968|
|   1|    [1]|   1| Angus Young|     Sydney|   1|       AC/DC|1973|
|   2| [1, 5]|   2|Eric Clapton|       NULL|NULL|        NULL|NULL|
|   3|    [3]|   3|Kirk Hammett|Los Angeles|   3|   Metallica|1981|
|NULL|   NULL|NULL|        NULL|  Liverpool|   4| The Beatles|1960|
+----+-------+----+------------+-----------+----+------------+----+



In [ ]:
# semi-joins
# only show rows from left table that has data in right table, but does not include right table data
# returns guitarists that have bands but no band data
guitarists_df.join(bands_df, join_condition, "left_semi").show()

+----+-------+---+------------+
|band|guitars| id|        name|
+----+-------+---+------------+
|   0|    [0]|  0|  Jimmy Page|
|   1|    [1]|  1| Angus Young|
|   3|    [3]|  3|Kirk Hammett|
+----+-------+---+------------+



In [ ]:
# left anti join
# only show rows from left table that DO NOT have data in right table, but does not include right table data
# returns eric clapton since he does not have a banda
guitarists_df.join(bands_df, join_condition, "left_anti").show()

+----+-------+---+------------+
|band|guitars| id|        name|
+----+-------+---+------------+
|   2| [1, 5]|  2|Eric Clapton|
+----+-------+---+------------+



In [ ]:
# things to bear in mind when doing joins
# this will crash because id is now ambigous after join because they both had id col
# guitarists_bands_df.select("id", "band").show

In [ ]:
# option 1 - rename the column on which we are joining
# if we rename the conflicting cols we wont have issues
guitarists_df.join(bands_df.withColumnRenamed("id", "band"), "band")

In [ ]:
# option 2 - drop the dupe column
# notice that we are doing the drop after join, this is allowed because
# spark remembers where we got the columns it knows bands df gave us id column after join
# spark keeps unique id for where columns came from
guitarists_bands_df.drop(bands_df.col("id"))

In [ ]:
# option 3 - rename the offending column and keep the data
# less attractive option because we still have duplicate data band and band id hold same data (duplicate)
bandsModDF = bands_df.withColumnRenamed("id", "bandId")
guitarists_df.join(bandsModDF, guitarists_df["band"] == bandsModDF["bandId"]).show()

+----+-------+---+------------+-----------+------+------------+----+
|band|guitars| id|        name|   hometown|bandId|        name|year|
+----+-------+---+------------+-----------+------+------------+----+
|   1|    [1]|  1| Angus Young|     Sydney|     1|       AC/DC|1973|
|   0|    [0]|  0|  Jimmy Page|     London|     0|Led Zeppelin|1968|
|   3|    [3]|  3|Kirk Hammett|Los Angeles|     3|   Metallica|1981|
+----+-------+---+------------+-----------+------+------------+----+



In [12]:
from pyspark.sql.functions import expr

# using complex types in joins
# guitars is an array in guitarists since guitarists have many guitars
# notice resulting table will have multiple rows for each guitarists for each guitar they own
guitarists_df.join(guitars_df.withColumnRenamed("id", "guitarId"), expr("array_contains(guitars, guitarId)")).show()


+----+-------+---+------------+--------------------+--------+------+------------+
|band|guitars| id|        name|          guitarType|guitarId|  make|       model|
+----+-------+---+------------+--------------------+--------+------+------------+
|   0|    [0]|  0|  Jimmy Page|Electric double-n...|       0|Gibson|    EDS-1275|
|   2| [1, 5]|  2|Eric Clapton|            Electric|       5|Fender|Stratocaster|
|   1|    [1]|  1| Angus Young|            Electric|       1|Gibson|          SG|
|   2| [1, 5]|  2|Eric Clapton|            Electric|       1|Gibson|          SG|
|   3|    [3]|  3|Kirk Hammett|            Electric|       3|   ESP|        M-II|
+----+-------+---+------------+--------------------+--------+------+------------+



In [ ]:
# Exercises
# 1. show all employees and their max salary they had
# 2. show all employees who were never managers
# 3. find job titles of the top 10 best employees of the company, max(toDate) is current job title
# 4. 

# Tips - try writing sql first then transfrom it to dataframe constructs
# - try to do intermediate dataframes and preview then step by step 

In [ ]:
# 1. show all employees and their max salary they had
# sub steps
# 1.1 pull in employees df 
# 1.2 pull in salaries df 
# 1.3 write how sql will look like
# 1.4 take max salary per employee
# 1.5 join with employees table

In [16]:
# 1.1 + 1.2
# pull in data 

employeesDF = ( 
    spark.read
    .format("jdbc")
    .option("driver", "org.postgresql.Driver")
    .option("user", "docker")
    .option("password", "docker")
    .option("dbtable", "public.employees")
    .option("url", "jdbc:postgresql://postgres:5432/rtjvm")
    .load()
)

salariesDF = ( 
    spark.read
    .format("jdbc")
    .option("driver", "org.postgresql.Driver")
    .option("user", "docker")
    .option("password", "docker")
    .option("dbtable", "public.salaries")
    .option("url", "jdbc:postgresql://postgres:5432/rtjvm")
    .load()
)

employeesDF.printSchema()
salariesDF.printSchema()

root
 |-- emp_no: integer (nullable = true)
 |-- birth_date: date (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- hire_date: date (nullable = true)

root
 |-- emp_no: integer (nullable = true)
 |-- salary: integer (nullable = true)
 |-- from_date: date (nullable = true)
 |-- to_date: date (nullable = true)



In [ ]:
# 1.3 write how sql will look like

max_salary_by employee as (
    
)

SELECT  e.emp_no, e.first_name, e.last_name, e.gender, MAX(s.salary) FROM e employees 
JOIN salary on s.emp_no = e.emp_no
GROUP BY e.emp_no, e.first_name, e.last_name, e.gender